In [1]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

import subprocess
import time

# Start the Ollama server in the background
print("Starting Ollama server...")
process = subprocess.Popen(["ollama", "serve"])
time.sleep(3)  # Give the server a few seconds to start up

# Pull the required embedding model
print("Pulling model 'nomic-embed-text'...")
!ollama pull nomic-embed-text
print("Done!")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...
Pulling model 'nomic-embed-text'...

Done!


In [2]:
# llm_text_minimal_split.py

import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA

import ollama

from google.colab import drive
drive.mount('/content/drive')

def load_json_lines(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().rstrip(",")
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def preprocess(df):
    df = df[
        [
            "review/profileName",
            "beer/beerId",
            "beer/ABV",
            "beer/style",
            "review/overall",
            "review/text",
        ]
    ].copy()

    df.columns = ["user", "item", "abv", "style", "rating", "text"]

    df["abv"] = pd.to_numeric(df["abv"], errors="coerce")
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    df["abv"] = df["abv"].fillna(df["abv"].median())
    df["style"] = df["style"].fillna("unknown")
    df["user"] = df["user"].fillna("unknown")
    df["item"] = df["item"].fillna("unknown")
    df["text"] = df["text"].fillna("")

    df = df.dropna(subset=["rating"]).reset_index(drop=True)

    return df


def clean_text(text):
    text = str(text).lower()
    text = text.replace("\t", " ")
    text = text.replace("\n", " ")
    return text[:800]


def get_ollama_embedding(text, model="nomic-embed-text", dim=768):
    text = clean_text(text)

    if not text or len(text.strip()) == 0:
        text = "empty review"

    for attempt in range(3):
        try:
            response = ollama.embed(
                model=model,
                input=text
            )

            if "embeddings" in response and len(response["embeddings"]) > 0:
                return response["embeddings"][0]

        except Exception as e:
            print(f"Ollama error attempt {attempt + 1}: {e}")

    print("Warning: empty embedding returned, using zeros.")
    return [0.0] * dim


def safe_transform(encoder, values):
    mapping = {label: idx for idx, label in enumerate(encoder.classes_)}
    return np.array([mapping.get(v, 0) for v in values])


def get_raw_embeddings(df, filename):
    if os.path.exists(filename):
        print("Loading raw embeddings:", filename)
        return np.load(filename)

    print("Generating raw embeddings:", filename)
    embeddings = [None] * len(df)

    def fetch_embedding(idx, text):
        emb = get_ollama_embedding(text)
        if len(emb) == 0:
            emb = [0.0] * 768
        return idx, emb

    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = {executor.submit(fetch_embedding, idx, text): idx for idx, text in enumerate(df["text"])}
        for future in tqdm(as_completed(futures), total=len(df), desc=f"Embedding {filename}"):
            idx, emb = future.result()
            embeddings[idx] = emb

    embeddings = np.array(embeddings, dtype=np.float32)
    np.save(filename, embeddings)

    return embeddings


print("Current working dir:", os.getcwd())

train_df = load_json_lines('/content/drive/MyDrive/SP26/CMPE 256/Group Project/new/beeradvocate_train.json')
val_df = load_json_lines('/content/drive/MyDrive/SP26/CMPE 256/Group Project/new/beeradvocate_val.json')
test_df = load_json_lines('/content/drive/MyDrive/SP26/CMPE 256/Group Project/new/beeradvocate_test.json')

print("Raw train:", train_df.shape)
print("Raw val:", val_df.shape)
print("Raw test:", test_df.shape)

train_df = preprocess(train_df)
val_df = preprocess(val_df)
test_df = preprocess(test_df)

SAMPLE_TRAIN = 50000
SAMPLE_VAL = 50000
SAMPLE_TEST = 50000

if len(train_df) > SAMPLE_TRAIN:
    train_df = train_df.sample(n=SAMPLE_TRAIN, random_state=42).reset_index(drop=True)

if len(val_df) > SAMPLE_VAL:
    val_df = val_df.sample(n=SAMPLE_VAL, random_state=42).reset_index(drop=True)

if len(test_df) > SAMPLE_TEST:
    test_df = test_df.sample(n=SAMPLE_TEST, random_state=42).reset_index(drop=True)

print("Used train:", train_df.shape)
print("Used val:", val_df.shape)
print("Used test:", test_df.shape)


# =========================
# Encoders fit on train only
# =========================

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()
style_encoder = LabelEncoder()

train_df["user_id"] = user_encoder.fit_transform(train_df["user"])
train_df["item_id"] = item_encoder.fit_transform(train_df["item"])
train_df["style_id"] = style_encoder.fit_transform(train_df["style"])

val_df["user_id"] = safe_transform(user_encoder, val_df["user"])
val_df["item_id"] = safe_transform(item_encoder, val_df["item"])
val_df["style_id"] = safe_transform(style_encoder, val_df["style"])

test_df["user_id"] = safe_transform(user_encoder, test_df["user"])
test_df["item_id"] = safe_transform(item_encoder, test_df["item"])
test_df["style_id"] = safe_transform(style_encoder, test_df["style"])

num_users = len(user_encoder.classes_)
num_items = len(item_encoder.classes_)
num_styles = len(style_encoder.classes_)

print("Users:", num_users)
print("Items:", num_items)
print("Styles:", num_styles)


# =========================
# Numeric feature: ABV only
# =========================

feature_cols = ["abv"]

scaler = StandardScaler()

train_numeric = scaler.fit_transform(train_df[feature_cols]).astype(np.float32)
val_numeric = scaler.transform(val_df[feature_cols]).astype(np.float32)
test_numeric = scaler.transform(test_df[feature_cols]).astype(np.float32)

train_ratings = train_df["rating"].values.astype(np.float32)
val_ratings = val_df["rating"].values.astype(np.float32)
test_ratings = test_df["rating"].values.astype(np.float32)


# =========================
# Ollama embeddings + PCA
# PCA fits on train only
# =========================

PCA_DIM = 32

train_raw = get_raw_embeddings(train_df, f"minimal_train_raw_emb_{SAMPLE_TRAIN}.npy")
val_raw = get_raw_embeddings(val_df, f"minimal_val_raw_emb_{SAMPLE_VAL}.npy")
test_raw = get_raw_embeddings(test_df, f"minimal_test_raw_emb_{SAMPLE_TEST}.npy")

train_pca_file = f"minimal_train_pca_emb_{SAMPLE_TRAIN}_dim{PCA_DIM}.npy"
val_pca_file = f"minimal_val_pca_emb_{SAMPLE_VAL}_dim{PCA_DIM}.npy"
test_pca_file = f"minimal_test_pca_emb_{SAMPLE_TEST}_dim{PCA_DIM}.npy"

if (
    os.path.exists(train_pca_file)
    and os.path.exists(val_pca_file)
    and os.path.exists(test_pca_file)
):
    print("Loading PCA embeddings...")
    train_text = np.load(train_pca_file)
    val_text = np.load(val_pca_file)
    test_text = np.load(test_pca_file)
else:
    print("Fitting PCA on train embeddings only...")
    pca = PCA(n_components=PCA_DIM, random_state=42)

    train_text = pca.fit_transform(train_raw).astype(np.float32)
    val_text = pca.transform(val_raw).astype(np.float32)
    test_text = pca.transform(test_raw).astype(np.float32)

    np.save(train_pca_file, train_text)
    np.save(val_pca_file, val_text)
    np.save(test_pca_file, test_text)

print("Train text shape:", train_text.shape)
print("Val text shape:", val_text.shape)
print("Test text shape:", test_text.shape)


class BeerDataset(Dataset):
    def __init__(self, df, numeric, text_emb, ratings):
        self.df = df
        self.numeric = numeric
        self.text_emb = text_emb
        self.ratings = ratings

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            "user": torch.tensor(self.df.iloc[idx]["user_id"], dtype=torch.long),
            "item": torch.tensor(self.df.iloc[idx]["item_id"], dtype=torch.long),
            "style": torch.tensor(self.df.iloc[idx]["style_id"], dtype=torch.long),
            "numeric": torch.tensor(self.numeric[idx], dtype=torch.float32),
            "text": torch.tensor(self.text_emb[idx], dtype=torch.float32),
            "rating": torch.tensor(self.ratings[idx], dtype=torch.float32),
        }


batch_size = 128

train_loader = DataLoader(
    BeerDataset(train_df, train_numeric, train_text, train_ratings),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    BeerDataset(val_df, val_numeric, val_text, val_ratings),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    BeerDataset(test_df, test_numeric, test_text, test_ratings),
    batch_size=batch_size,
    shuffle=False
)


class HybridLLMRecommender(nn.Module):
    def __init__(
        self,
        num_users,
        num_items,
        num_styles,
        numeric_dim,
        text_dim,
        user_emb_dim=32,
        item_emb_dim=32,
        style_emb_dim=16,
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(num_users, user_emb_dim)
        self.item_embedding = nn.Embedding(num_items, item_emb_dim)
        self.style_embedding = nn.Embedding(num_styles, style_emb_dim)

        input_dim = user_emb_dim + item_emb_dim + style_emb_dim + numeric_dim + text_dim

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, user, item, style, numeric, text):
        user_vec = self.user_embedding(user)
        item_vec = self.item_embedding(item)
        style_vec = self.style_embedding(style)

        x = torch.cat([user_vec, item_vec, style_vec, numeric, text], dim=1)

        raw = self.mlp(x).squeeze(1)

        return 1 + 4 * torch.sigmoid(raw)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = HybridLLMRecommender(
    num_users=num_users,
    num_items=num_items,
    num_styles=num_styles,
    numeric_dim=train_numeric.shape[1],
    text_dim=train_text.shape[1],
).to(device)

criterion = nn.SmoothL1Loss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0003,
    weight_decay=1e-4
)


def evaluate(loader):
    model.eval()

    preds = []
    labels = []

    with torch.no_grad():
        for batch in loader:
            user = batch["user"].to(device)
            item = batch["item"].to(device)
            style = batch["style"].to(device)
            numeric = batch["numeric"].to(device)
            text = batch["text"].to(device)
            rating = batch["rating"].to(device)

            output = model(user, item, style, numeric, text)

            preds.extend(output.cpu().numpy())
            labels.extend(rating.cpu().numpy())

    rmse = np.sqrt(mean_squared_error(labels, preds))
    mae = mean_absolute_error(labels, preds)

    return rmse, mae


epochs = 30
patience = 7
bad_epochs = 0
best_val_rmse = float("inf")

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        user = batch["user"].to(device)
        item = batch["item"].to(device)
        style = batch["style"].to(device)
        numeric = batch["numeric"].to(device)
        text = batch["text"].to(device)
        rating = batch["rating"].to(device)

        optimizer.zero_grad()

        output = model(user, item, style, numeric, text)
        loss = criterion(output, rating)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_rmse, val_mae = evaluate(val_loader)

    print(
        f"Epoch {epoch + 1} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Val RMSE: {val_rmse:.4f} | "
        f"Val MAE: {val_mae:.4f}"
    )

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        bad_epochs = 0
        torch.save(model.state_dict(), "best_minimal_llm_model.pt")
        print("Best model saved.")
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("Early stopping.")
        break


model.load_state_dict(torch.load("best_minimal_llm_model.pt"))

test_rmse, test_mae = evaluate(test_loader)

print("=" * 40)
print("Best Val RMSE:", best_val_rmse)
print("Final Test RMSE:", test_rmse)
print("Final Test MAE:", test_mae)
print("=" * 40)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working dir: /content
Raw train: (1269292, 13)
Raw val: (158661, 13)
Raw test: (158662, 13)
Used train: (50000, 6)
Used val: (50000, 6)
Used test: (50000, 6)
Users: 8327
Items: 12170
Styles: 104
Generating raw embeddings: minimal_train_raw_emb_50000.npy


Embedding minimal_train_raw_emb_50000.npy: 100%|██████████| 50000/50000 [04:56<00:00, 168.65it/s]


Generating raw embeddings: minimal_val_raw_emb_50000.npy


Embedding minimal_val_raw_emb_50000.npy: 100%|██████████| 50000/50000 [04:38<00:00, 179.83it/s]


Generating raw embeddings: minimal_test_raw_emb_50000.npy


Embedding minimal_test_raw_emb_50000.npy: 100%|██████████| 50000/50000 [04:38<00:00, 179.44it/s]


Fitting PCA on train embeddings only...
Train text shape: (50000, 32)
Val text shape: (50000, 32)
Test text shape: (50000, 32)
Using device: cuda
Epoch 1 | Loss: 0.2672 | Val RMSE: 0.6438 | Val MAE: 0.4798
Best model saved.
Epoch 2 | Loss: 0.1970 | Val RMSE: 0.5623 | Val MAE: 0.4182
Best model saved.
Epoch 3 | Loss: 0.1648 | Val RMSE: 0.5280 | Val MAE: 0.3969
Best model saved.
Epoch 4 | Loss: 0.1504 | Val RMSE: 0.5126 | Val MAE: 0.3876
Best model saved.
Epoch 5 | Loss: 0.1422 | Val RMSE: 0.5073 | Val MAE: 0.3843
Best model saved.
Epoch 6 | Loss: 0.1373 | Val RMSE: 0.4991 | Val MAE: 0.3788
Best model saved.
Epoch 7 | Loss: 0.1338 | Val RMSE: 0.5015 | Val MAE: 0.3783
Epoch 8 | Loss: 0.1294 | Val RMSE: 0.4944 | Val MAE: 0.3734
Best model saved.
Epoch 9 | Loss: 0.1257 | Val RMSE: 0.4969 | Val MAE: 0.3744
Epoch 10 | Loss: 0.1237 | Val RMSE: 0.4914 | Val MAE: 0.3710
Best model saved.
Epoch 11 | Loss: 0.1215 | Val RMSE: 0.4899 | Val MAE: 0.3688
Best model saved.
Epoch 12 | Loss: 0.1191 | Val 